In [ ]:
'''
This notebook is used to analyse activities individually, based on a particular project (hence, a particular scenario over ther years).
'''

# Libraries

In [ ]:
# Import BW25 packages. You'll notice the packages are imported individually, unlike a one-and-done import with BW2.
import bw2data as bd
# import bw2io as bi
import bw2calc as bc
import bw2analyzer as bwa
import matplotlib.pyplot as plt
from collections import defaultdict
# import numpy as np
# import seaborn as sns
import pandas as pd

# 1. Support Functions

In [ ]:
from config import recipe_midpoint_h, activities_li, reference_product_lithium, activities_ni, reference_product_nickel, activities_mn, reference_product_manganese

## 1.1. Functions

In [ ]:
def search_and_get_exchanges(database, search_term):
    """
    Search for an activity in the database using the search term, 
    and return all exchanges for the first matching activity.

    Parameters:
    database: The Brightway2 database object to search within.
    search_term: The term to search for in the database.

    Returns:
    list: A list of exchanges for the first activity found using the search term.
    """
    # Perform the search
    search_results = database.search(search_term)
    
    # Get the first result
    if search_results:
        first_result = search_results[0]
        
        # Get all exchanges
        exchanges = list(first_result.exchanges())
        return exchanges
    else:
        return None

In [ ]:
def filter_and_display_elementary_flows(exchanges):
    """
    Filters the exchanges to include only elementary flows and prints each one.

    Parameters:
    exchanges: A list of exchanges to filter.

    Returns:
    list: A list of elementary flows filtered from the exchanges.
    """
    # Filter for elementary flows
    elementary_flows = [ex for ex in exchanges if ex['type'] == 'biosphere']

    # Display the elementary flows
    for flow in elementary_flows:
        print(flow)
    
    return elementary_flows

In [ ]:
def find_activity_by_name_product_location(db_name, activity_name, reference_product=None, location=None):
    """
    Find an activity by name, reference product, and location in a given database.
    
    Parameters:
    - db_name: The name of the database to search.
    - activity_name: The name of the activity to search for.
    - reference_product: The reference product to filter the search (optional).
    - location: The location to filter the search (optional).
    
    Returns:
    - activity: The activity object found in the database.
    """
    db = bd.Database(db_name)
    search_results = db.search(activity_name)
    
    if not search_results:
        raise ValueError(f"Activity '{activity_name}' not found in database '{db_name}'")
    
    # Filter by reference product if provided
    if reference_product:
        search_results = [
            act for act in search_results if act['reference product'] == reference_product
        ]
        
        if not search_results:
            raise ValueError(f"No activity found with reference product '{reference_product}' for '{activity_name}'")
    
    # Filter by location if provided
    if location:
        search_results = [
            act for act in search_results if act['location'] == location
        ]
        
        if not search_results:
            raise ValueError(f"No activity found with location '{location}' for '{activity_name}' and '{reference_product}'")
    
    # Return the first match that has the correct filters applied
    return search_results[0]



In [ ]:
def traverse_exchanges(activity, depth=0, max_depth=10, visited=None):
    """
    Recursively traverse the exchanges of an activity and its inputs.

    Parameters:
    - activity: The starting activity.
    - depth: Current depth of recursion.
    - max_depth: Maximum depth to explore.
    - visited: Set of visited activity keys to avoid cycles.
    """
    if visited is None:
        visited = set()
    indent = '  ' * depth
    activity_key = activity.key
    if activity_key in visited:
        print(f"{indent}Activity '{activity['name']}' ({activity['location']}) already visited. Skipping to avoid cycles.")
        return
    visited.add(activity_key)
    print(f"{indent}Activity: {activity['name']} ({activity['location']})")
    if depth >= max_depth:
        print(f"{indent}Maximum depth reached. Not exploring further.")
        return
    for exc in activity.exchanges():
        if exc['type'] == 'technosphere':
            input_activity = exc.input
            print(f"{indent}  Input: {input_activity['name']} ({input_activity['location']}) | Amount: {exc['amount']}")
            # Recursively explore inputs
            traverse_exchanges(input_activity, depth=depth+1, max_depth=max_depth, visited=visited)
        elif exc['type'] == 'biosphere':
            biosphere_flow = exc.input
            print(f"{indent}  Biosphere Flow: {biosphere_flow['name']} | Amount: {exc['amount']}")
        else:
            print(f"{indent}  Other Exchange: {exc.input} | Amount: {exc['amount']}")


In [ ]:
def run_comprehensive_lcia(activity, methods_list):
    """
    Perform a comprehensive LCIA for a given activity across multiple impact categories.
    
    Parameters:
    - activity: The specific activity for which the LCIA is to be performed.
    - methods_list: A list of tuples representing the impact assessment methods.

    Returns:
    - lca_results: A dictionary with methods as keys and their corresponding LCIA scores as values.
    """

    # Define the functional unit (e.g., 1 unit of the activity)
    functional_unit = {activity: 1}

    # Initialize a dictionary to store the results
    lca_results = defaultdict(float)

    # Loop over all impact categories in the method
    for method in methods_list:
        # Run LCI and LCIA
        lca = bc.LCA(functional_unit, method)
        lca.lci()
        lca.lcia()
        
        # Store the result for each category
        lca_results[method] = lca.score

    # Output the results
    for category, score in lca_results.items():
        print(f"{category}: {score}")
    
    return lca_results

In [ ]:
def run_lcia_across_years(activity_name, methods_list, databases):
    """
    Run LCIA across multiple years (databases) and store results.
    
    Parameters:
    - activity_name: The name of the activity to analyze.
    - methods_list: A list of tuples representing the impact assessment methods.
    - databases: A list of database names (one for each year).
    
    Returns:
    - results_dict: A dictionary where keys are years and values are LCIA results dictionaries.
    """

    results_dict = {}

    for db_name in databases:
        # Set the current project
        bd.projects.set_current(project_name)

        try:
            # Find the specific activity in the current database
            activity = find_activity_by_name_product_location(db_name, activity_name)
        except ValueError as e:
            print(e)
            continue
    
        # Extract the year by finding the part that looks like a year

        # Urgently need to find a better way to do this:

        year = next(part for part in db_name.split('_') if part.isdigit() and len(part) == 4)

        # Run LCIA for the current year
        lca_results = run_comprehensive_lcia(activity, methods_list)

        # Store the results indexed by year
        results_dict[year] = lca_results

    return results_dict

In [ ]:
def run_lcia_for_multiple_activities(activities, methods_list, databases):
    """
    Run LCIA across multiple years for multiple activities and store results.
    
    Parameters:
    - activities: A list of activity names to analyze.
    - methods_list: A list of tuples representing the impact assessment methods.
    - databases: A list of database names (one for each year).
    
    Returns:
    - results_dict: A nested dictionary where the first key is the activity name,
      the second key is the year, and the value is the LCIA results dictionary.
    """

    results_dict = {}

    for activity_name in activities:
        print(f"Processing activity: {activity_name}")
        lcia_results_per_year = run_lcia_across_years(activity_name, methods_list, databases)
        results_dict[activity_name] = lcia_results_per_year

    return results_dict

## 1.2. Visuals

In [ ]:
def plot_lcia_results_across_years(results_dict, method):
    """
    Plot LCIA results for a specific impact category across multiple years.
    
    Parameters:
    - results_dict: A dictionary where keys are years and values are LCIA results dictionaries.
    - method: The specific impact category tuple to plot (e.g., ('ReCiPe Midpoint (H)', 'climate change', 'GWP100')).
    """

    years = sorted(results_dict.keys())
    scores = [results_dict[year][method] for year in years]

    plt.figure(figsize=(10, 6))
    plt.plot(years, scores, marker='o', linestyle='-', color='b')
    
    plt.xlabel('Year')
    plt.ylabel('Impact Score')
    plt.title(f'LCIA Results for {method[1]} ({method[2]}) across Years')
    
    plt.grid(True)
    plt.show()

In [ ]:
def plot_all_lcia_results_across_years(lcia_results_per_year, methods_list):
    """
    Loop through all methods in the list and plot the LCIA results across years for each method.
    
    Parameters:
    - lcia_results_per_year: A dictionary where keys are years and values are LCIA results dictionaries.
    - methods_list: A list of tuples representing the impact assessment methods.
    """

    for method in methods_list:
        # Plot LCIA results for each method
        plot_lcia_results_across_years(lcia_results_per_year, method)

In [ ]:
def plot_combined_lcia_results_across_years(lcia_results_per_year, methods_list, activity_name, project_name):
    """
    Plot all LCIA results for different methods across years into a single image using subplots.
    
    Parameters:
    - lcia_results_per_year: A dictionary where keys are years and values are LCIA results dictionaries.
    - methods_list: A list of tuples representing the impact assessment methods.
    - activity_name: The name of the activity being analyzed.
    - project_name: The name of the project.
    """

    num_methods = len(methods_list)
    num_cols = 3  # Number of columns for subplots
    num_rows = (num_methods + num_cols - 1) // num_cols  # Calculate the number of rows needed

    plt.figure(figsize=(15, 5 * num_rows))

    for i, method in enumerate(methods_list, 1):
        years = sorted(lcia_results_per_year.keys())
        scores = [lcia_results_per_year[year][method] for year in years]

        plt.subplot(num_rows, num_cols, i)
        plt.plot(years, scores, marker='o', linestyle='-', color='b')

        plt.xlabel('Year')
        plt.ylabel('Impact Score')
        plt.title(f'{method[1]} ({method[2]})')

        plt.grid(True)

    # Add a main title that includes the activity name and project name
    plt.suptitle(f'LCIA Results for {activity_name}\nProject: {project_name}', fontsize=16, y=1.02)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_comparison_across_activities(results_dict, methods_list, activities):
    """
    Plot comparison of LCIA results across activities for each impact category,
    with a single, wrapping legend at the bottom of the figure.
    
    Parameters:
    - results_dict: A nested dictionary with LCIA results for all activities and years.
    - methods_list: A list of tuples representing the impact assessment methods.
    - activities: A list of activity names that were analyzed.
    """

    num_methods = len(methods_list)
    num_cols = 3  # Number of columns for subplots
    num_rows = (num_methods + num_cols - 1) // num_cols  # Calculate the number of rows needed

    plt.figure(figsize=(15, 5 * num_rows))

    for i, method in enumerate(methods_list, 1):
        plt.subplot(num_rows, num_cols, i)
        for activity_name in activities:
            years = sorted(results_dict[activity_name].keys())
            scores = [results_dict[activity_name][year][method] for year in years]

            plt.plot(years, scores, marker='o', linestyle='-', label=activity_name)

        plt.xlabel('Year')
        plt.ylabel('Impact Score')
        plt.title(f'{method[1]} ({method[2]})')

        plt.grid(True)

    # Add a wrapping legend for all plots at the bottom
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.figlegend(handles, labels, loc='upper center', ncol=1, fontsize='large')

    plt.suptitle(f'Comparison of LCIA Results Across Activities\nProject: SSP1-Base', fontsize=16, y=1.02)
    plt.tight_layout()  # Adjust layout to make space for the legend at the bottom
    plt.show()

## 1.3. Lists

In [ ]:
'''# List of ReCiPe Midpoint (H)
# H stands for Hierarchist
## https://www.rivm.nl/bibliotheek/rapporten/2016-0104.pdf
recipe_midpoint_h = [
    ('ReCiPe Midpoint (H)', 'terrestrial ecotoxicity', 'TETPinf'),
    ('ReCiPe Midpoint (H)', 'natural land transformation', 'NLTP'),
    ('ReCiPe Midpoint (H)', 'photochemical oxidant formation', 'POFP'),
    ('ReCiPe Midpoint (H)', 'human toxicity', 'HTPinf'),
    ('ReCiPe Midpoint (H)', 'marine eutrophication', 'MEP'),
    ('ReCiPe Midpoint (H)', 'climate change', 'GWP100'),
    ('ReCiPe Midpoint (H)', 'particulate matter formation', 'PMFP'),
    ('ReCiPe Midpoint (H)', 'agricultural land occupation', 'ALOP'),
    ('ReCiPe Midpoint (H)', 'freshwater eutrophication', 'FEP'),
    ('ReCiPe Midpoint (H)', 'metal depletion', 'MDP'),
    ('ReCiPe Midpoint (H)', 'terrestrial acidification', 'TAP100'),
    ('ReCiPe Midpoint (H)', 'water depletion', 'WDP'),
    ('ReCiPe Midpoint (H)', 'urban land occupation', 'ULOP'),
    ('ReCiPe Midpoint (H)', 'ionising radiation', 'IRP_HE'),
    ('ReCiPe Midpoint (H)', 'fossil depletion', 'FDP'),
    ('ReCiPe Midpoint (H)', 'freshwater ecotoxicity', 'FETPinf'),
    ('ReCiPe Midpoint (H)', 'marine ecotoxicity', 'METPinf'),
    ('ReCiPe Midpoint (H)', 'ozone depletion', 'ODPinf')
]'''

In [ ]:
recipe_endpoint_h_a = [
    ('ReCiPe Endpoint (H,A)', 'human health', 'total'),
    ('ReCiPe Endpoint (H,A)', 'ecosystem quality', 'total'),
    ('ReCiPe Endpoint (H,A)', 'resources', 'total')
]

In [ ]:
databases_remindSSP1_baseline = [
 'EI38_cutoff_remind_SSP1-Base_2020_baseline',
 'EI38_cutoff_remind_SSP1-Base_2025_baseline',
 'EI38_cutoff_remind_SSP1-Base_2030_baseline',
 'EI38_cutoff_remind_SSP1-Base_2035_baseline',
 'EI38_cutoff_remind_SSP1-Base_2040_baseline',
 'EI38_cutoff_remind_SSP1-Base_2045_baseline',
 'EI38_cutoff_remind_SSP1-Base_2050_baseline'
]

databases_remindSSP1_VSI_test_1 = [
 'EI38_cutoff_remind_SSP1-Base_2020_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2025_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2030_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2035_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2040_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2045_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2050_VSI_test_1' 
]

databases_remindSSP1_energy = [
 'EI38_cutoff_remind_SSP1-Base_2020_energy',
 'EI38_cutoff_remind_SSP1-Base_2025_energy',
 'EI38_cutoff_remind_SSP1-Base_2030_energy',
 'EI38_cutoff_remind_SSP1-Base_2035_energy',
 'EI38_cutoff_remind_SSP1-Base_2040_energy',
 'EI38_cutoff_remind_SSP1-Base_2045_energy',
 'EI38_cutoff_remind_SSP1-Base_2050_energy' 
]

databases_remindSSP1_energy_and_VSI_test_1 = [
 'EI38_cutoff_remind_SSP1-Base_2020_energy_and_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2025_energy_and_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2030_energy_and_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2035_energy_and_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2040_energy_and_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2045_energy_and_VSI_test_1',
 'EI38_cutoff_remind_SSP1-Base_2050_energy_and_VSI_test_1' 
]


In [ ]:
'''activities_li = [
    ('lithium carbonate production, from concentrated brine', 'GLO'),
    ('lithium carbonate production, from spodumene', 'RoW'),
    ('lithium carbonate production, from spodumene', 'CN')
]

reference_product_lithium = 'lithium carbonate'

activities_ni = [
    ('treatment of metal part of electronics scrap, in copper, anode, by electrolytic refining', 'RoW'),
    ('platinum group metal mine operation, ore with high palladium content', 'RU'),
    ('platinum group metal, extraction and refinery operations', 'ZA'),
    ('processing of nickel-rich materials', 'GLO'),
    ('smelting and refining of nickel concentrate, 16% Ni', 'GLO'),
    ('smelting and refining of nickel concentrate, 7% Ni', 'CN'),
    ('treatment of metal part of electronics scrap, in copper, anode, by electrolytic refining', 'SE'),
    ('cobalt production', 'GLO')
]

reference_product_nickel = 'nickel, class 1'

activities_mn = [
    ('manganese dioxide production', 'GLO'),
    ('manganese sulfate production', 'GLO')
]

reference_product_manganese = 'manganese sulfate''''

# 2. Imports and Declarations

In [ ]:
#Creating/accessing the project
bd.projects.set_current("LNV-EI38-20250414")

In [ ]:
bd.databases

In [ ]:
# Accessing the SSP1 2020 database
db_name = 'EI38_cutoff_remind_SSP1-Base_2020_energy'
db = bd.Database(db_name)

# 3. Lithium Carbonate

In [ ]:
'''
The ecoinvent 3.8 database has an activity "market for lithium carbonate".
Such activity feeds from:
- lithium carbonate production, from spodumene (CN)
- lithium carbonate production, from spodumene (RoW)
- lithium carbonate production, from concentrated brine (GLO)
- treatment of used Li-ion battery, hydrometallurgical treatment (GLO)
- [ ... ]  Other activities
When applying the PREMISE2.0 SSP1-Base update based on energy, we get a new activity:
- market for lithium hydroxide, battery grade
'''

## 3.1. LiO2 production, battery grade

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.

search_term = "lithium hydroxide production, battery grade"
reference_product = reference_product_lithium

In [ ]:
db_name.search(search_term)

In [ ]:
activity = find_activity_by_name_product(db_name, search_term)

In [ ]:
exchanges = search_and_get_exchanges(db, search_term)
exchanges

In [ ]:
elementary_flows = filter_and_display_elementary_flows(exchanges)

In [ ]:
'''
Let's explore the activiies:
'lithium carbonate production, from Salar de Atacama'
'lithium carbonate production, from Salar de Olaroz'
'lithium carbonate production, from Salar de Cauchari-Olaroz'
'lithium carbonate production, from Salar del Hombre Muerto'
'lithium carbonate production, from Chaerhan salt lake'
'''

### 3.1.1. lithium carbonate production, from Salar de Atacama

#### 3.1.1.1. Getting activity and travesing inventory

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
activity_name = 'lithium carbonate production, from Salar de Atacama'
activity = find_activity_by_name_product(db_name, activity_name)

In [ ]:
# To view the inventory
for exc in activity.exchanges():
    print(f"Input: {exc.input} | Amount: {exc['amount']}")

In [ ]:
bwa.print_recursive_supply_chain(activity, max_level=3, cutoff=0.03)

In [ ]:
# Traverse the exchanges up to a desired depth (e.g., max_depth=2)
traverse_exchanges(activity, max_depth=5)

In [ ]:
def collect_exchanges(activity, depth=0, max_depth=5, visited=None, data=None):
    """
    Recursively traverse the exchanges of an activity and its inputs,
    collecting data into a list of exchanges.

    Parameters:
    - activity: The starting activity.
    - depth: Current depth of recursion.
    - max_depth: Maximum depth to explore.
    - visited: Set of visited activity keys to avoid cycles.
    - data: List to collect the data.

    Returns:
    - data: A list of dictionaries containing exchange information.
    """
    if visited is None:
        visited = set()
    if data is None:
        data = []
    activity_key = activity.key
    if activity_key in visited:
        return data
    visited.add(activity_key)

    if depth >= max_depth:
        return data

    for exc in activity.exchanges():
        if exc['type'] == 'technosphere':
            input_activity = exc.input
            # Collect data about the exchange
            data.append({
                'depth': depth + 1,
                'from_activity_name': activity['name'],
                'from_activity_location': activity['location'],
                'from_reference_product': activity.get('reference product', ''),
                'from_unit': activity.get('unit', ''),
                'from_activity_type': activity.get('type', ''),
                'to_activity_name': input_activity['name'],
                'to_activity_location': input_activity['location'],
                'to_reference_product': input_activity.get('reference product', ''),
                'to_unit': input_activity.get('unit', ''),
                'to_activity_type': input_activity.get('type', ''),
                'amount': exc['amount'],
                'exchange_type': exc['type'],
            })
            # Recursively explore inputs
            collect_exchanges(input_activity, depth=depth+1, max_depth=max_depth, visited=visited, data=data)
        elif exc['type'] == 'biosphere':
            biosphere_flow = exc.input
            # Collect data about the biosphere flow
            data.append({
                'depth': depth + 1,
                'from_activity_name': activity['name'],
                'from_activity_location': activity['location'],
                'from_reference_product': activity.get('reference product', ''),
                'from_unit': activity.get('unit', ''),
                'from_activity_type': activity.get('type', ''),
                'to_activity_name': biosphere_flow['name'],
                'to_activity_location': biosphere_flow.get('location', ''),
                'to_reference_product': '',
                'to_unit': biosphere_flow.get('unit', ''),
                'to_activity_type': 'biosphere',
                'amount': exc['amount'],
                'exchange_type': exc['type'],
            })
        else:
            # Handle other exchange types if necessary
            pass

    return data

In [ ]:
# Collect exchanges data with desired max depth
exchanges_data = collect_exchanges(activity, max_depth=3)

# Convert the data to a DataFrame
exchanges_df = pd.DataFrame(exchanges_data)

In [ ]:
exchanges_df

In [ ]:
exchanges_df.to_csv('exchanges_data.csv', index=False)

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

### 1.1.2. lithium carbonate production, from Salar de Olaroz

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
activity_name = 'lithium carbonate production, from Salar de Olaroz'

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

### 1.1.3. lithium carbonate production, from Salar de Cauchari-Olaroz

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
activity_name = 'lithium carbonate production, from Salar de Cauchari-Olaroz'

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

### 1.1.4. lithium carbonate production, from Salar del Hombre Muerto

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
activity_name = 'lithium carbonate production, from Salar del Hombre Muerto'

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

### 1.1.5. lithium carbonate production, from Chaerhan salt lake

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
activity_name = 'lithium carbonate production, from Chaerhan salt lake'

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

## 3.2. All compared

In [ ]:
# Run LCIA for all activities across all years
lcia_results_all_activities = run_lcia_for_multiple_activities(activities_li, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
# Plot comparison of results across activities
plot_comparison_across_activities(lcia_results_all_activities, recipe_midpoint_h, activities_li)

# 4. Nickel

## 4.1. nickel, class 1

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
activity_name = 'nickel, class 1'

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
# search_term = "market for nickel, class 1"

In [ ]:
# exchanges = search_and_get_exchanges(db, search_term)
# exchanges

In [ ]:
# elementary_flows = filter_and_display_elementary_flows(exchanges)

In [ ]:
'''
Let's explore the following exchanges:
Exchange: 0.161162933095465 kilogram 'platinum group metal mine operation, ore with high palladium content' (kilogram, RU, None) to 'market for nickel, class 1' (kilogram, GLO, None)>,
Exchange: 0.0309617633971047 kilogram 'platinum group metal, extraction and refinery operations' (kilogram, ZA, None) to 'market for nickel, class 1' (kilogram, GLO, None)>,
Exchange: 0.139645918974656 kilogram 'processing of nickel-rich materials' (kilogram, GLO, None) to 'market for nickel, class 1' (kilogram, GLO, None)>,
Exchange: 0.161885636831319 kilogram 'smelting and refining of nickel concentrate, 16% Ni' (kilogram, GLO, None) to 'market for nickel, class 1' (kilogram, GLO, None)>,
Exchange: 0.167667266718152 kilogram 'smelting and refining of nickel concentrate, 7% Ni' (kilogram, CN, None) to 'market for nickel, class 1' (kilogram, GLO, None)>,
Exchange: 0.00202310190815179 kilogram 'treatment of metal part of electronics scrap, in copper, anode, by electrolytic refining' (kilogram, RoW, None) to 'market for nickel, class 1' (kilogram, GLO, None)>,
Exchange: 0.336648016439843 kilogram 'cobalt production' (kilogram, GLO, None) to 'market for nickel, class 1' (kilogram, GLO, None)>]
'''


### 4.1.1.

## 4.2. All compared across years

In [ ]:
# Run LCIA for all activities across all years
lcia_results_all_activities = run_lcia_for_multiple_activities(activities_ni, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
# Plot comparison of results across activities
plot_comparison_across_activities(lcia_results_all_activities, recipe_midpoint_h, activities_ni)

# 5. Manganese concentrate

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
# search_term = "market for manganese sulfate"


'''
When looking at NMC-811 hydroxide, we can see "manganese sulfate".
By exploring manganese sulfate, we can see:
-manganese dioxide production
-manganese sulfate production
both generating manganese sulfate

'''

In [ ]:
# exchanges = search_and_get_exchanges(db, search_term)
# exchanges

## 5.1. Manganese Sulfate

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
activity_name = "market for manganese sulfate"

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

### 5.1.1. Manganese dioxide production

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
activity_name = 'manganese dioxide production'

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

### 5.1.1. Manganese sulfate production

In [ ]:
# Search term from updated SSP1-Base2020 database.
# Updated from:
## Source: Jarod C. Kelly, Michael Wang, Qiang Dai, Olumide Winjobi, Energy, greenhouse gas, and water life cycle analysis of lithium carbonate and lithium hydroxide monohydrate from brine and ore resources and their use in lithium ion battery cathodes and lithium ion batteries, Resources, Conservation and Recycling, Volume 174, 2021, 105762, ISSN 0921-3449, https://doi.org/10.1016/j.resconrec.2021.105762.
activity_name = 'manganese sulfate production'

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")

## 5.2. All compared across years

In [ ]:
# Run LCIA for all activities across all years
lcia_results_all_activities = run_lcia_for_multiple_activities(activities_mn, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
# Plot comparison of results across activities
plot_comparison_across_activities(lcia_results_all_activities, recipe_midpoint_h, activities_mn)

# 6. Cobalt

# 7. battery cell, NMC-811

In [ ]:
activity_name = 'battery cell, NMC-811'

In [ ]:
# Run the LCIA across all years
lcia_results_per_year = run_lcia_across_years(activity_name, recipe_midpoint_h, databases_remindSSP1_energy)

In [ ]:
plot_combined_lcia_results_across_years(lcia_results_per_year, recipe_midpoint_h, activity_name, "SSP1-Base (Electricity Updated)")